# Stage 1 — Data Ingestion

**SkillMap** analyzes job posting data to find skill combinations tied to compensation, discover job role archetypes, and predict salary tier.

This notebook is the first pipeline stage. It:
1. Loads every raw CSV table from the three dataset folders in `data/raw/`
2. Prints the shape, column names, and sample rows of each table
3. Checks data types and missing values
4. Checks that the key columns listed in `CLAUDE.md` are present
5. Writes a combined exploration summary to `outputs/01_summary.txt`

Raw files are read-only and are never modified here.

## Step 1 — Imports and paths

All paths are built with `pathlib.Path`. The project root is found by walking up from the working directory until `CLAUDE.md` is found, so the notebook works whether Jupyter was started from the project root or from `notebooks/`.

In [ ]:
import io
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)


def find_project_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUTPUT_DIR / "01_summary.txt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DIR}")

## Step 2 — Table registry

Each dataset is a folder in `data/raw/` holding one or more CSV tables (see `CLAUDE.md`). Every table is listed with its path relative to `data/raw/` and the key columns it should have, so schema changes show up early.

The multi-file datasets are **not merged here**. Joins on `job_link` or `job_id` happen in Stage 2.

Memory controls:
- `NROWS` caps rows for every table (e.g. `100_000` for a quick run). `None` loads everything.
- Tables larger than `LARGE_FILE_MB` on disk load only the first `LARGE_FILE_NROWS` rows. This covers `job_summary.csv` (4.8 GB of description text), which would need well over 10 GB of RAM to load fully. The summary file records which tables were capped.

In [ ]:
DATASET_DESCRIPTIONS = {
    "linkedin_jobs": "1.3M LinkedIn Jobs & Skills 2024",
    "linkedin_postings": "LinkedIn Job Postings 2023-2024",
    "ds_salaries": "Data Science Salaries 2020-2025",
}

TABLES = {
    # linkedin_jobs — join key: job_link
    "linkedin_jobs/linkedin_job_postings": {
        "file": "linkedin_jobs/linkedin_job_postings.csv",
        "key_columns": ["job_link", "job_title", "company", "job_location", "job_level", "job_type"],
    },
    "linkedin_jobs/job_skills": {
        "file": "linkedin_jobs/job_skills.csv",
        "key_columns": ["job_link", "job_skills"],
    },
    "linkedin_jobs/job_summary": {
        "file": "linkedin_jobs/job_summary.csv",
        "key_columns": ["job_link", "job_summary"],
    },
    # linkedin_postings — join keys: job_id, company_id
    "linkedin_postings/postings": {
        "file": "linkedin_postings/postings.csv",
        "key_columns": ["job_id", "title", "min_salary", "max_salary", "pay_period",
                        "formatted_experience_level", "company_id", "location"],
    },
    "linkedin_postings/salaries": {
        "file": "linkedin_postings/jobs/salaries.csv",
        "key_columns": ["job_id", "min_salary", "max_salary", "pay_period"],
    },
    "linkedin_postings/job_skills": {
        "file": "linkedin_postings/jobs/job_skills.csv",
        "key_columns": ["job_id", "skill_abr"],
    },
    "linkedin_postings/job_industries": {
        "file": "linkedin_postings/jobs/job_industries.csv",
        "key_columns": ["job_id", "industry_id"],
    },
    "linkedin_postings/benefits": {
        "file": "linkedin_postings/jobs/benefits.csv",
        "key_columns": ["job_id", "type"],
    },
    "linkedin_postings/companies": {
        "file": "linkedin_postings/companies/companies.csv",
        "key_columns": ["company_id", "name", "company_size"],
    },
    "linkedin_postings/company_industries": {
        "file": "linkedin_postings/companies/company_industries.csv",
        "key_columns": ["company_id", "industry"],
    },
    "linkedin_postings/company_specialities": {
        "file": "linkedin_postings/companies/company_specialities.csv",
        "key_columns": ["company_id", "speciality"],
    },
    "linkedin_postings/employee_counts": {
        "file": "linkedin_postings/companies/employee_counts.csv",
        "key_columns": ["company_id", "employee_count"],
    },
    "linkedin_postings/skills_map": {
        "file": "linkedin_postings/mappings/skills.csv",
        "key_columns": ["skill_abr", "skill_name"],
    },
    "linkedin_postings/industries_map": {
        "file": "linkedin_postings/mappings/industries.csv",
        "key_columns": ["industry_id", "industry_name"],
    },
    # ds_salaries
    "ds_salaries/salaries": {
        "file": "ds_salaries/DataScience_salaries_2025.csv",
        "key_columns": ["job_title", "salary_in_usd", "company_size", "experience_level",
                        "employment_type", "remote_ratio"],
    },
}
for name, spec in TABLES.items():
    spec["description"] = DATASET_DESCRIPTIONS[name.split("/")[0]]

NROWS = None                 # e.g. 100_000 for a quick development run
LARGE_FILE_MB = 1024         # tables bigger than this on disk are capped...
LARGE_FILE_NROWS = 200_000   # ...to this many rows (set to None to force a full load)

## Step 3 — Load the CSV files

Each file is loaded with `low_memory=False` so pandas infers one dtype per column instead of mixing dtypes across chunks. A missing file is reported and skipped, so the other datasets can still be explored.

In [ ]:
capped_tables = {}


def load_table(name: str, spec: dict):
    """Load one raw CSV described by `spec`, applying row caps; return None if missing."""
    path = RAW_DIR / spec["file"]
    if not path.exists():
        print(f"[MISSING] {name}: {path} not found — skipping")
        return None
    size_mb = path.stat().st_size / 1024**2
    nrows = NROWS
    if size_mb > LARGE_FILE_MB and LARGE_FILE_NROWS is not None:
        nrows = LARGE_FILE_NROWS if nrows is None else min(nrows, LARGE_FILE_NROWS)
        capped_tables[name] = nrows
    df = pd.read_csv(path, low_memory=False, nrows=nrows)
    note = f" [capped at first {nrows:,} rows]" if name in capped_tables else ""
    print(f"[LOADED]  {name}: {df.shape[0]:,} rows x {df.shape[1]} cols ({size_mb:,.1f} MB on disk){note}")
    return df


dataframes = {}
for name, spec in TABLES.items():
    df = load_table(name, spec)
    if df is not None:
        dataframes[name] = df

if not dataframes:
    raise FileNotFoundError(f"No tables found in {RAW_DIR}. Place the dataset folders there and re-run.")

## Step 4 — Shape and column names

In [ ]:
for name, df in dataframes.items():
    print(f"=== {name} ({TABLES[name]['description']}) ===")
    print(f"Shape: {df.shape}")
    print(f"Columns ({len(df.columns)}): {list(df.columns)}\n")

## Step 5 — Sample rows

Five random rows (fixed seed) from each dataset. Random rows show more variety than `head()`, which often shows only the first few postings from one scrape.

In [ ]:
for name, df in dataframes.items():
    print(f"=== {name} ===")
    display(df.sample(n=min(5, len(df)), random_state=RANDOM_STATE))

## Step 6 — Data types

`df.info()` gives dtypes, non-null counts, and deep memory usage. Watch for columns that should be numeric (e.g. salary) but loaded as `object`. That usually means stray text such as currency symbols or ranges, which Stage 2 must clean.

In [ ]:
for name, df in dataframes.items():
    print(f"=== {name} ===")
    df.info(memory_usage="deep")
    print()

## Step 7 — Missing values

Null count and percentage per column, sorted by the most missing. Salary columns are the main concern: Stage 2 fills them with the median grouped by `job_title` and `experience_level`, so heavy missingness there limits how well that works.

In [ ]:
def null_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return per-column null counts and percentages, sorted descending."""
    counts = df.isna().sum()
    report = pd.DataFrame({
        "null_count": counts,
        "null_pct": (counts / len(df) * 100).round(2),
        "dtype": df.dtypes.astype(str),
    })
    return report.sort_values("null_count", ascending=False)


null_reports = {}
for name, df in dataframes.items():
    null_reports[name] = null_report(df)
    print(f"=== {name} ===")
    display(null_reports[name])

## Step 8 — Duplicates and key-column check

This counts fully duplicated rows, which Stage 2 will drop. It also checks each dataset against the key columns listed in `CLAUDE.md`. Any missing key column needs a rename in Stage 2 before later stages can use it.

In [ ]:
def duplicate_count(df: pd.DataFrame) -> int:
    """Count fully duplicated rows, hashing unhashable cells (e.g. lists) as strings."""
    try:
        return int(df.duplicated().sum())
    except TypeError:
        return int(df.astype(str).duplicated().sum())


integrity = {}
for name, df in dataframes.items():
    expected = TABLES[name]["key_columns"]
    missing_cols = [c for c in expected if c not in df.columns]
    integrity[name] = {"duplicates": duplicate_count(df), "missing_key_columns": missing_cols}

integrity_df = pd.DataFrame(integrity).T
integrity_df

## Step 9 — Numeric summary

Descriptive statistics for numeric columns. For salary fields, check the min and max for outliers such as zero salaries, hourly rates mixed with annual ones, or values in non-USD currencies.

In [ ]:
for name, df in dataframes.items():
    numeric = df.select_dtypes(include=np.number)
    print(f"=== {name} ===")
    if numeric.empty:
        print("(no numeric columns)\n")
    else:
        display(numeric.describe().T)

## Step 10 — Save the exploration summary

The results above are written to `outputs/01_summary.txt` as one plain-text report for later stages and the project write-up.

In [ ]:
def build_summary(dataframes: dict) -> str:
    """Build a plain-text exploration summary covering every loaded dataset."""
    buf = io.StringIO()
    line = "=" * 80
    buf.write(f"SkillMap — Stage 1 Data Ingestion Summary\n")
    buf.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}\n")
    buf.write(f"Row limit (NROWS): {NROWS if NROWS else 'full load'}\n")
    if capped_tables:
        capped = ", ".join(f"{n} (first {r:,} rows)" for n, r in capped_tables.items())
        buf.write(f"Large tables loaded partially: {capped}\n")
    missing_files = [spec["file"] for n, spec in TABLES.items() if n not in dataframes]
    buf.write(f"Tables loaded: {len(dataframes)}/{len(TABLES)}")
    buf.write(f" (missing: {', '.join(missing_files)})\n" if missing_files else "\n")

    for name, df in dataframes.items():
        spec = TABLES[name]
        buf.write(f"\n{line}\n{name} — {spec['description']} ({spec['file']})\n{line}\n")
        buf.write(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
        buf.write(f"Duplicate rows: {integrity[name]['duplicates']:,}\n")
        missing_cols = integrity[name]["missing_key_columns"]
        buf.write(f"Missing expected key columns: {missing_cols if missing_cols else 'none'}\n")

        buf.write("\n-- Columns, dtypes, and nulls --\n")
        buf.write(null_reports[name].to_string())
        buf.write("\n")

        numeric = df.select_dtypes(include=np.number)
        if not numeric.empty:
            buf.write("\n-- Numeric summary --\n")
            buf.write(numeric.describe().T.to_string())
            buf.write("\n")

        buf.write("\n-- Sample rows (random_state=42) --\n")
        sample = df.sample(n=min(5, len(df)), random_state=RANDOM_STATE)
        # to_string() ignores display.max_colwidth, so truncate long text cells explicitly
        sample = sample.apply(lambda col: col.map(lambda v: str(v).replace("\n", " ")[:40]))
        buf.write(sample.to_string())
        buf.write("\n")
    return buf.getvalue()


summary_text = build_summary(dataframes)
SUMMARY_PATH.write_text(summary_text, encoding="utf-8")
print(f"Summary written to {SUMMARY_PATH} ({len(summary_text):,} characters)")

## Next steps

- Check `integrity_df` for **missing key columns**. Any schema change there needs handling in `src/preprocessing.py`.
- Stage 2 joins: `linkedin_jobs` tables on `job_link`; `linkedin_postings` tables on `job_id` / `company_id`, with `mappings/` for skill and industry names.
- `postings.csv` salaries mix hourly, monthly, and yearly `pay_period` values. Use `normalized_salary` or annualize before tiering.
- Use the null report to decide which columns Stage 2 should drop and which should be filled.
- Continue to `02_preprocessing.ipynb`.